# Canonical SMILES conversion and deduplication

Standardise each file's SMILES to canonical SMILES and remove duplicates.

In [3]:
import pandas as pd
import os
from rdkit import Chem

data_dir = r'../data/in_vitro_data'
output_dir = r'../data/in_vitro_data_deduplicated'

# Create the output directory
os.makedirs(output_dir, exist_ok=True)

files = [
    'chembl_CL_raw.csv',
    'chembl_Fu_raw.csv',
    'biogen_CL_raw.csv',
    'chembl_PPB_raw.csv',
    'tdc_cl_hep.csv',
    'tdc_cl_mic.csv',
    'tdc_half_life.csv',
    'tdc_PPBR_AZ.csv'
]

smiles_column_candidates = ['smiles', 'SMILES', 'Smiles', 'Drug']

def get_canonical_smiles(smiles_str):
    """Convert SMILES to canonical SMILES"""
    if pd.isna(smiles_str):
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return Chem.MolToSmiles(mol)
        else:
            return None
    except:
        return None

print("Processing files...\n")

summary_data = []

for file in files:
    file_path = os.path.join(data_dir, file)
    
    # Load the file
    try:
        df = pd.read_csv(file_path)
        original_rows = len(df)
        
        # Find the SMILES column
        smiles_col = None
        for col in smiles_column_candidates:
            if col in df.columns:
                smiles_col = col
                break
        
        if smiles_col:
            # Convert to canonical SMILES
            print(f"Processing {file}...")
            df['canonical_smiles'] = df[smiles_col].apply(get_canonical_smiles)
            
            # Number of rows with invalid SMILES
            invalid_smiles = df['canonical_smiles'].isna().sum()
            
            # Keep valid rows only
            df_valid = df[df['canonical_smiles'].notna()].copy()
            
            # Deduplicate on canonical SMILES (keep the first occurrence)
            df_deduplicated = df_valid.drop_duplicates(subset=['canonical_smiles'], keep='first')
            
            final_rows = len(df_deduplicated)
            removed_rows = original_rows - final_rows
            
            # Replace the original smiles column with canonical_smiles
            df_deduplicated[smiles_col] = df_deduplicated['canonical_smiles']
            df_deduplicated = df_deduplicated.drop('canonical_smiles', axis=1)
            
            # Save the file
            output_path = os.path.join(output_dir, file.replace('.csv', '_deduplicated.csv'))
            df_deduplicated.to_csv(output_path, index=False)
            
            summary_data.append({
                'File': file,
                'Original Rows': original_rows,
                'Invalid SMILES': invalid_smiles,
                'After Dedup': final_rows,
                'Removed': removed_rows,
                'Dedup %': f"{(removed_rows/original_rows)*100:.2f}%" if original_rows > 0 else "N/A"
            })
            
            print(f"  ✓ {original_rows} → {final_rows} rows (removed {removed_rows})\n")
        else:
            print(f"  ✗ {file}: No SMILES column found\n")
            
    except Exception as e:
        print(f"  ✗ {file}: Error - {e}\n")

Processing files...

Processing chembl_CL_raw.csv...
  ✓ 80447 → 44278 rows (removed 36169)

Processing chembl_Fu_raw.csv...
  ✓ 14033 → 7248 rows (removed 6785)

Processing biogen_CL_raw.csv...
  ✓ 3521 → 3521 rows (removed 0)

Processing chembl_PPB_raw.csv...
  ✓ 13778 → 8268 rows (removed 5510)

Processing tdc_cl_hep.csv...
  ✓ 1213 → 1020 rows (removed 193)

Processing tdc_cl_mic.csv...
  ✓ 1102 → 1102 rows (removed 0)

Processing tdc_half_life.csv...
  ✓ 667 → 665 rows (removed 2)

Processing tdc_PPBR_AZ.csv...
  ✓ 1614 → 1614 rows (removed 0)



## Summary

In [4]:
# Summary table
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*100)
print("Deduplication Summary")
print("="*100)
print(summary_df.to_string(index=False))

print("\n" + "="*100)
print(f"Output Directory: {output_dir}")
print("="*100)

# Overall statistics
total_original = summary_df['Original Rows'].sum()
total_removed = summary_df['Removed'].sum()
total_final = summary_df['After Dedup'].sum()

print(f"\nOverall statistics:")
print(f"  Original rows: {total_original}")
print(f"  Removed rows: {total_removed}")
print(f"  Final rows: {total_final}")
print(f"  Removal rate: {(total_removed/total_original)*100:.2f}%")


Deduplication Summary
              File  Original Rows  Invalid SMILES  After Dedup  Removed Dedup %
 chembl_CL_raw.csv          80447               0        44278    36169  44.96%
 chembl_Fu_raw.csv          14033               0         7248     6785  48.35%
 biogen_CL_raw.csv           3521               0         3521        0   0.00%
chembl_PPB_raw.csv          13778               0         8268     5510  39.99%
    tdc_cl_hep.csv           1213               0         1020      193  15.91%
    tdc_cl_mic.csv           1102               0         1102        0   0.00%
 tdc_half_life.csv            667               0          665        2   0.30%
   tdc_PPBR_AZ.csv           1614               0         1614        0   0.00%

Output Directory: ..\data\in_vitro_data_deduplicated

Overall statistics:
  Original rows: 116375
  Removed rows: 48659
  Final rows: 67716
  Removal rate: 41.81%
